# 01 · Define & Explore — the conserved RBD epitope + the binder metrics

**Standard slot:** *define & explore.* **For Project 07 this means:** clean the RBD target, pick a
**conserved, ACE2-facing (neutralizing) epitope**, fix the metrics, and run a tiny mock binder
batch as your hello-world (D0).

> **Defensive framing.** Every design here is steered to *block* the virus at the ACE2 face. Enhancing
> viral affinity/escape/fitness is out of scope (`README.md` → Responsible research, `MASTER_BLUEPRINT.md §7`).

Run `00_setup.ipynb` first in this session.

## The metrics, precisely (binder design)

| Metric | Means | Does **not** mean |
|--------|-------|-------------------|
| `pae_interaction` (AF2-Multimer) | interface confidence — the key binder metric (lower = better) | measured affinity |
| interface pLDDT | local confidence at the interface | stability / K_D |
| scRMSD | designed-vs-predicted binder backbone self-consistency | binding |
| shape complementarity | packing across the interface | function |
| **worst-case breadth pae** | does the binder hold across ALL variant RBDs? | best-case is not breadth |


## Setup paths

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

## Target prep + epitope choice (EXAMPLE — verify on the real structure)

Clean the RBD (chain selection, remove ACE2), then choose hotspots on the **conserved ACE2-binding
face**. The residues below are **EXAMPLE placeholders** — derive the real ones from 6M0J + a
sarbecovirus conservation analysis (see `data/README.md`).

In [ ]:
import binder_tools as bt

TARGET = "RBD"
# EXAMPLE conserved-epitope hotspots (SARS-CoV-2 numbering) — VERIFY from 6M0J + conservation.
HOTSPOTS = bt.parse_hotspots("E417,E484,E501")
print("target:", TARGET, "| EXAMPLE hotspots (verify):", HOTSPOTS)

## Hello-world: a tiny mock binder batch

Develop the plumbing with the deterministic `mock` backend (no GPU). Switch `tool=` to the real
backends on an A100 (see `MANUAL.md §2`). **Mock numbers are SYNTHETIC — never report them.**

In [ ]:
designs = bt.generate_binders_bindcraft(TARGET, HOTSPOTS, n=5, tool="mock")
bt.score_designs(designs, tool="mock")
d = designs[0]
print("example:", d.design_id, "len=", d.length,
      "pae_interaction=", d.pae_interaction, "scrmsd=", d.scrmsd, "sc=", d.shape_complementarity)
print("ACE2-footprint overlap (competition proxy):", bt.hotspot_overlap(d.contact_residues, HOTSPOTS))
print("[reminder] every number above is SYNTHETIC (mock).")

## Breadth concept: epitope conservation across variants `[core]`

A neutralizing binder is variant-resistant only if its epitope is **conserved**. The snippet below is
**EXAMPLE_DATA** (toy aligned RBD fragments) to demonstrate `epitope_conservation()`; in your project
you align a real variant panel (e.g. with MAFFT) and score the columns under your epitope.

In [ ]:
# EXAMPLE_DATA — toy aligned RBD fragments (NOT real sequences), positions 1..10 for illustration.
EXAMPLE_VARIANTS = {
    "Wuhan":   "NITNLCPFGE",
    "Delta":   "NITNLCPFGE",
    "Omicron": "NITNLCPFGK",   # a change at position 10
}
conserved_epitope = [3, 5, 7]   # toy positions; high conservation
variable_epitope  = [10]        # toy position; low conservation
print("conserved-epitope conservation:", bt.epitope_conservation(conserved_epitope, EXAMPLE_VARIANTS))
print("variable-epitope conservation: ", bt.epitope_conservation(variable_epitope, EXAMPLE_VARIANTS))
print("[EXAMPLE_DATA] toy demonstration of the breadth rationale — replace with the real panel.")

## D0 checklist
- [ ] Conserved-epitope map (conservation across variants) + justification of the chosen face.
- [ ] One-paragraph definition of each binder metric **with** its 'does not mean' note.
- [ ] One reproduced mock mini-run (designs scored, breadth rationale shown).
- [ ] `LOG.md` entry: tool version, GPU, seed.

**Next:** `02_generate.ipynb` — the two-paradigm campaign at the conserved epitope.